# 7. Оценка через EvaluationOrchestrator

**Цель:** показать API `EvaluationCase`/`EvaluationOrchestrator` на одном известном случае — синтетическом одномерном ряде с размеченным onset `is_extreme`. Оркестратор запускает тот же `GraphEVTPipeline` и считает задержку обнаружения относительно истинного времени события. Как и в ноутбуке 4, baseline здесь заканчивается точно перед известным onset — это oracle-assisted выбор, допустимый только потому, что onset этого синтетического ряда известен, и неприменимый к неразмеченным данным.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, EvaluationCase, EvaluationOrchestrator, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/generated/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 1
true_onset = int(np.flatnonzero(event_mask)[0])
true_onset

604

In [2]:
# Oracle-assisted baseline: the known onset keeps the baseline free of the injected event.
BASELINE_SIZE = true_onset
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.95, alarm_probability=0.999, persistence=3),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
    verbose=True,
)
evaluation = EvaluationOrchestrator(pipeline, early_tolerance=0, k=3, verbose=True).evaluate([
    EvaluationCase(x, event_time=true_onset, case_id="fractal 1-D"),
])

print("Known onset:", true_onset)
print("Summary:", evaluation.summary())
pd.DataFrame(evaluation.to_records())

Evaluating cases:   0%|          | 0/1 [00:00<?, ?it/s]

Known onset: 604
Summary: {'1-d': {'cases': 1, 'hit': 1, 'miss': 0, 'early_alarm': 0, 'false_alarm': 0, 'correct_rejection': 0, 'unlabelled': 0, 'detection_rate': 1.0, 'false_alarm_rate': nan, 'mean_delay': 15.0, 'median_delay': 15.0}}


,case_id,route,outcome,stop_reason,detected_time,true_event_time,delay,true_source,heuristic_source,heuristic_rank,gnn_source,gnn_rank,gat_source,gat_rank
0,fractal 1-D,1-d,hit,single_channel_cannot_localize,619,604,15,None,None,None,None,None,None,None


Один случай не образует benchmark: он проверяет, что `EvaluationOrchestrator` исполняет pipeline и корректно считает задержку/раннюю тревогу относительно единственной известной истины, не более того.

**Самопроверка:** почему `early_tolerance=0` здесь — осознанный выбор, а не default? Что произойдёт, если вместо oracle-assisted baseline использовать первые 75% ряда, как в ноутбуке 3? Почему совпадение с одной известной истиной не переносится на неразмеченные данные?